# Lightweight CNN Benchmark + Grad-CAM for Citrus Leaf Health Screening
**Dataset:** IndiaAI AIKosh – *Citrus Plant Leaves Dataset* (Healthy vs Unhealthy)

**What this notebook does (run cells top to bottom):**
1. Finds the images and counts them per class
2. Removes duplicate / near-duplicate images (important – duplicates inflate accuracy)
3. Trains 5 lightweight pretrained CNNs with stratified 5-fold cross-validation
4. Reports Accuracy, Precision, Recall, F1, AUC (mean ± SD) + parameters, model size, CPU inference time
5. Makes Grad-CAM heatmaps for the best model
6. Saves every table and figure for the paper to `OUT_DIR`

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [ ]:
!pip install -q imagehash

import os, time, random, json, copy, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imagehash
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Settings – EDIT `DATA_DIR`
Download the dataset zip from AIKosh, upload it to Google Drive, unzip it, and put the folder path here.
The code searches inside `DATA_DIR` for the folders named *Healthy* and *Unhealthy* automatically.

In [ ]:
# --- In Colab, mount Drive first ---
from google.colab import drive; drive.mount('/content/drive')

DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive/Citrus_Plant_Leaves")
OUT_DIR  = os.environ.get("OUT_DIR",  "/content/drive/MyDrive/citrus_results")
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"   # leave as 0 for the real run
PRETRAINED = os.environ.get("PRETRAINED", "1") == "1"   # keep 1 (ImageNet transfer learning)

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 2 if QUICK_TEST else 20
PATIENCE   = 5            # early stopping on validation loss
LR         = 1e-4
N_FOLDS    = 2 if QUICK_TEST else 5
PHASH_DIST = 4            # Hamming distance <= this = near-duplicate
MODEL_NAMES = ["mobilenet_v2", "efficientnet_b0", "shufflenet_v2", "resnet18", "squeezenet1_1"]
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Find images and count per class

In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

def find_class_dirs(root):
    """Return {class_name: folder} for the deepest folders that directly contain images."""
    found = {}
    for d, _, files in os.walk(root):
        if any(f.lower().endswith(EXTS) for f in files):
            found[os.path.basename(d)] = d
    return found

class_dirs = find_class_dirs(DATA_DIR)
print("Image folders found:", {k: v for k, v in class_dirs.items()})
assert len(class_dirs) == 2, "Expected exactly 2 image folders (Healthy / Unhealthy) – check DATA_DIR"

# Fixed label order: Healthy = 0, Unhealthy = 1 (Unhealthy is the 'positive' class)
CLASSES = sorted(class_dirs, key=lambda c: (0 if c.lower().startswith("healthy") else 1, c))
paths, labels = [], []
for i, c in enumerate(CLASSES):
    for f in sorted(os.listdir(class_dirs[c])):
        if f.lower().endswith(EXTS):
            paths.append(os.path.join(class_dirs[c], f)); labels.append(i)

counts_raw = pd.Series(labels).map(dict(enumerate(CLASSES))).value_counts()
print("\nImages per class (raw):\n", counts_raw)

# Image size statistics (for the 'Dataset description' section)
sizes = []
for p in paths:
    with Image.open(p) as im: sizes.append(im.size)
sizes = np.array(sizes)
print(f"\nWidth  min/median/max: {sizes[:,0].min()} / {int(np.median(sizes[:,0]))} / {sizes[:,0].max()}")
print(f"Height min/median/max: {sizes[:,1].min()} / {int(np.median(sizes[:,1]))} / {sizes[:,1].max()}")

## 4. Duplicate / near-duplicate removal (perceptual hash)
Report these numbers in the paper – reviewers value this.

In [ ]:
hashes = []
for p in paths:
    with Image.open(p) as im: hashes.append(imagehash.phash(im.convert("RGB")))

keep, removed = [], []
kept_hashes = []
for i, h in enumerate(hashes):
    dup_of = next((j for j, kh in kept_hashes if (h - kh) <= PHASH_DIST), None)
    if dup_of is None:
        keep.append(i); kept_hashes.append((i, h))
    else:
        removed.append({"removed": paths[i], "duplicate_of": paths[dup_of],
                        "cross_class": labels[i] != labels[dup_of]})

pd.DataFrame(removed).to_csv(f"{OUT_DIR}/duplicates_removed.csv", index=False)
n_cross = sum(r["cross_class"] for r in removed)
print(f"Removed {len(removed)} duplicate/near-duplicate images "
      f"({n_cross} of them had a copy in the OTHER class – label conflicts).")

paths  = [paths[i] for i in keep]
labels = np.array([labels[i] for i in keep])
counts_clean = pd.Series(labels).map(dict(enumerate(CLASSES))).value_counts()
print("\nImages per class (after cleaning):\n", counts_clean)

pd.DataFrame({"raw": counts_raw, "after_dedup": counts_clean}).to_csv(f"{OUT_DIR}/table1_dataset_counts.csv")

if QUICK_TEST:   # tiny subset just to check the code runs
    idx = np.concatenate([np.where(labels == k)[0][:12] for k in (0, 1)])
    paths, labels = [paths[i] for i in idx], labels[idx]

# Sample images figure
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for r, k in enumerate((0, 1)):
    for c, i in enumerate(np.where(labels == k)[0][:4]):
        axes[r, c].imshow(Image.open(paths[i]).convert("RGB")); axes[r, c].axis("off")
        axes[r, c].set_title(CLASSES[k], fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_sample_images.png", dpi=300); plt.show()

## 5. Data pipeline

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class LeafDS(Dataset):
    def __init__(self, paths, labels, tf): self.p, self.y, self.tf = paths, labels, tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), int(self.y[i])

## 6. Models (ImageNet-pretrained, last layer replaced for 2 classes)

In [ ]:
def build_model(name, n_classes=2):
    W = "DEFAULT" if PRETRAINED else None
    if name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=W); m.classifier[1] = nn.Linear(m.last_channel, n_classes)
        cam_layer = m.features[-1]
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=W); m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        cam_layer = m.features[-1]
    elif name == "shufflenet_v2":
        m = models.shufflenet_v2_x1_0(weights=W); m.fc = nn.Linear(m.fc.in_features, n_classes)
        cam_layer = m.conv5
    elif name == "resnet18":
        m = models.resnet18(weights=W); m.fc = nn.Linear(m.fc.in_features, n_classes)
        cam_layer = m.layer4
    elif name == "squeezenet1_1":
        m = models.squeezenet1_1(weights=W); m.classifier[1] = nn.Conv2d(512, n_classes, 1)
        cam_layer = m.features[-1]
    else:
        raise ValueError(name)
    return m, cam_layer

def model_stats(name):
    m, _ = build_model(name); m.eval()
    params = sum(p.numel() for p in m.parameters())
    tmp = f"/tmp/{name}.pt"; torch.save(m.state_dict(), tmp)
    size_mb = os.path.getsize(tmp) / 1e6
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        for _ in range(3): m(x)                       # warm-up
        n = 5 if QUICK_TEST else 50
        t0 = time.perf_counter()
        for _ in range(n): m(x)
    cpu_ms = (time.perf_counter() - t0) / n * 1000
    return {"params_M": params / 1e6, "size_MB": size_mb, "cpu_ms_per_image": cpu_ms}

## 7. Training with stratified 5-fold cross-validation

In [ ]:
def run_epoch(model, loader, crit, opt=None):
    train = opt is not None
    model.train(train)
    tot_loss, ys, probs = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x); loss = crit(out, y)
            if train: opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * len(y)
            ys.append(y.cpu().numpy()); probs.append(torch.softmax(out, 1)[:, 1].detach().cpu().numpy())
    return tot_loss / len(loader.dataset), np.concatenate(ys), np.concatenate(probs)

def metrics(y, p, thr=0.5):
    yhat = (p >= thr).astype(int)
    return {"accuracy": accuracy_score(y, yhat),
            "precision": precision_score(y, yhat, zero_division=0),
            "recall": recall_score(y, yhat, zero_division=0),
            "f1": f1_score(y, yhat, zero_division=0),
            "auc": roc_auc_score(y, p) if len(set(y)) > 1 else float("nan")}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(paths, labels))
fold_rows, curves, oof = [], {}, {}
NW = 0 if QUICK_TEST else 2

for name in MODEL_NAMES:
    oof[name] = np.zeros(len(labels)); curves[name] = []
    for k, (tr, va) in enumerate(folds):
        set_seed(SEED + k)
        tr_dl = DataLoader(LeafDS([paths[i] for i in tr], labels[tr], train_tf), BATCH_SIZE, shuffle=True, num_workers=NW)
        va_dl = DataLoader(LeafDS([paths[i] for i in va], labels[va], eval_tf), BATCH_SIZE, num_workers=NW)
        model, _ = build_model(name); model.to(DEVICE)
        # class-weighted loss handles any imbalance
        w = torch.tensor(len(tr) / (2 * np.bincount(labels[tr], minlength=2)), dtype=torch.float32, device=DEVICE)
        crit = nn.CrossEntropyLoss(weight=w)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        best, best_state, wait, hist = 1e9, None, 0, []
        for ep in range(EPOCHS):
            trl, _, _ = run_epoch(model, tr_dl, crit, opt)
            val, yv, pv = run_epoch(model, va_dl, crit)
            hist.append({"epoch": ep + 1, "train_loss": trl, "val_loss": val, "val_acc": metrics(yv, pv)["accuracy"]})
            if val < best: best, best_state, wait = val, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= PATIENCE: break
        model.load_state_dict(best_state)
        _, yv, pv = run_epoch(model, va_dl, crit)
        oof[name][va] = pv
        m = metrics(yv, pv); m.update({"model": name, "fold": k + 1, "epochs_run": len(hist)})
        fold_rows.append(m); curves[name].append(hist)
        print(f"{name:16s} fold {k+1}: acc={m['accuracy']:.4f} f1={m['f1']:.4f} auc={m['auc']:.4f} ({len(hist)} ep)")
        if k == 0: torch.save(best_state, f"{OUT_DIR}/{name}_fold1.pt")   # used for Grad-CAM

fold_df = pd.DataFrame(fold_rows); fold_df.to_csv(f"{OUT_DIR}/per_fold_results.csv", index=False)

## 8. Results table (Table 2 in the paper)

In [ ]:
cols = ["accuracy", "precision", "recall", "f1", "auc"]
summary = fold_df.groupby("model")[cols].agg(["mean", "std"])
stats = pd.DataFrame({n: model_stats(n) for n in MODEL_NAMES}).T
table = pd.DataFrame(index=MODEL_NAMES)
for c in cols:
    table[c] = [f"{summary.loc[n,(c,'mean')]*100:.2f} ± {summary.loc[n,(c,'std')]*100:.2f}" for n in MODEL_NAMES]
table["params (M)"] = stats["params_M"].round(2)
table["size (MB)"] = stats["size_MB"].round(1)
table["CPU ms/image"] = stats["cpu_ms_per_image"].round(1)
table.to_csv(f"{OUT_DIR}/table2_model_comparison.csv")
print(table.to_string())

best_model = summary[("f1", "mean")].idxmax()
print("\nBest model by mean F1:", best_model)
json.dump({"best_model": best_model, "classes": CLASSES, "n_images": int(len(labels)),
           "duplicates_removed": len(removed), "cross_class_duplicates": int(n_cross)},
          open(f"{OUT_DIR}/run_info.json", "w"), indent=2)

## 9. Figures: accuracy vs model size, confusion matrix, training curves

In [ ]:
# Fig 2 – accuracy vs model size (the 'lightweight trade-off' figure)
plt.figure(figsize=(6, 4.5))
for n in MODEL_NAMES:
    plt.scatter(stats.loc[n, "size_MB"], summary.loc[n, ("accuracy", "mean")] * 100, s=80)
    plt.annotate(n, (stats.loc[n, "size_MB"], summary.loc[n, ("accuracy", "mean")] * 100),
                 textcoords="offset points", xytext=(5, 5), fontsize=9)
plt.xlabel("Model size (MB)"); plt.ylabel("Mean accuracy (%)"); plt.grid(alpha=.3)
plt.title("Accuracy vs model size"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig2_accuracy_vs_size.png", dpi=300); plt.show()

# Fig 3 – confusion matrix of best model (out-of-fold predictions, all images)
cm = confusion_matrix(labels, (oof[best_model] >= 0.5).astype(int))
plt.figure(figsize=(4, 3.6)); plt.imshow(cm, cmap="Greens")
for i in range(2):
    for j in range(2): plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=13)
plt.xticks([0, 1], CLASSES); plt.yticks([0, 1], CLASSES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix – {best_model}")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_confusion_matrix.png", dpi=300); plt.show()

# Fig 4 – training curves of best model, fold 1
h = pd.DataFrame(curves[best_model][0])
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend()
ax[1].plot(h.epoch, h.val_acc * 100); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Validation accuracy (%)")
plt.suptitle(f"Training curves – {best_model} (fold 1)"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig4_training_curves.png", dpi=300); plt.show()

## 10. Grad-CAM for the best model (Fig 5)
Heatmaps show which leaf regions drove the prediction. Ask a plant pathologist to look at these.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0]))
    def __call__(self, x, cls=None):
        self.model.zero_grad(); out = self.model(x)
        cls = out.argmax(1).item() if cls is None else cls
        out[0, cls].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * self.acts).sum(1)).squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, cls, torch.softmax(out, 1)[0, cls].item()

model, layer = build_model(best_model)
model.load_state_dict(torch.load(f"{OUT_DIR}/{best_model}_fold1.pt", map_location="cpu"))
model.eval()
for mod in model.modules():            # in-place ReLU breaks backward hooks
    if isinstance(mod, nn.ReLU): mod.inplace = False
cam_fn = GradCAM(model, layer)

va_idx = folds[0][1]
show = np.concatenate([[i for i in va_idx if labels[i] == k][:3] for k in (0, 1)])
fig, axes = plt.subplots(2, len(show), figsize=(3 * len(show), 6))
for c, i in enumerate(show):
    img = Image.open(paths[i]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    cam, pred, conf = cam_fn(eval_tf(img).unsqueeze(0))
    cam = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255
    axes[0, c].imshow(img); axes[0, c].set_title(f"True: {CLASSES[labels[i]]}", fontsize=9)
    axes[1, c].imshow(img); axes[1, c].imshow(cam, cmap="jet", alpha=0.45)
    axes[1, c].set_title(f"Pred: {CLASSES[pred]} ({conf:.2f})", fontsize=9)
    axes[0, c].axis("off"); axes[1, c].axis("off")
plt.suptitle(f"Grad-CAM – {best_model}"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig5_gradcam.png", dpi=300); plt.show()

print("\nAll outputs saved in:", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print("  ", f)